# 1.2 怎樣留下真正沒練過的題目？


如果學生練習「貓看狗。」後，考試又出完全相同的一句，高分可能只表示背熟了。要檢查新材料，先把文字分成不同用途，再讓模型接觸其中的一部分。

本節的材料有五筆文字，其中「貓看狗。」重複一次：

`貓看狗。／狗看貓。／鳥看魚。／魚看鳥。／貓看狗。`

希望得到三份互不重複的材料。**訓練資料**用來調整模型內的可調數字，這些數字叫參數；**驗證資料**用來選設定；**測試資料**留到設定選完後，作最後檢查。例如在驗證資料上比較看一字與看三字的模型，就是選設定；測試資料這時仍保留。

專案的 `split_documents` 先移除完全一樣的文字，再打亂與分組。這五筆只有四份不同文字，結果會是訓練兩份、驗證一份、測試一份。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
from tiny_perceptron.data import split_documents

docs = ["貓看狗。", "狗看貓。", "鳥看魚。", "魚看鳥。", "貓看狗。"]
parts = split_documents(docs, seed=42)
for name, documents in parts.items():
    print(name, len(documents), documents)
assert not (set(parts["train"]) & set(parts["validation"]))

每行輸出都列出組名、份數與實際文字，先逐句核對沒有同一句跨組。`seed=42` 固定打亂順序的起點，方便重現這次分組；42 本身沒有語言意義。最後的 `&` 找訓練與驗證共有的文字，`not` 要求它為空。

只查字串完全相等仍不夠。若先把「貓看狗。狗看貓。」切成視窗，再把相鄰的「貓看狗。」與「看狗。狗」分到兩組，它們共享「看狗。」。模型已練過其中的連續內容，卻又被當作新材料考一次，這叫**資料洩漏**。所以通常先按完整來源分組，再在各組內切片；同一篇的改寫、同一段對話的版本也應一起分。

共用「貓、看、狗」這些字沒有問題：新文章本來就使用舊字。需要避免的是把同一來源的複本或高度重疊片段當成新題。

練習再加一份「貓看狗。」。不同材料總數應仍是四；把新增那份改成「鳥看貓。」，總數才變五。這次沒有訓練模型，只是在確認以後的評分能回答「遇到新材料會怎樣」。

<details>
<summary>補充與重做</summary>

執行環境見 [W.1](https://birdhackor.github.io/tiny-perceptron-vlm/first-steps.html#W.1)，`.items()`、集合與字典見 [W.2](https://birdhackor.github.io/tiny-perceptron-vlm/first-steps.html#W.2)。本例的斷言只查訓練與驗證之間的完全重複，不是所有來源洩漏的自動證明。反覆看驗證結果會影響選擇，因此設定挑好後仍需保留的測試資料。

</details>
